In [3]:
!{sys.executable} -m pip install torch --index-url https://download.pytorch.org/whl/cpu

Looking in indexes: https://download.pytorch.org/whl/cpu
   ---------------------------------------- 0.0/124.0 MB ? eta -:--:--
   ---------------------------------------- 0.3/124.0 MB ? eta -:--:--
    --------------------------------------- 1.6/124.0 MB 7.7 MB/s eta 0:00:16
   - -------------------------------------- 4.2/124.0 MB 9.7 MB/s eta 0:00:13
   - -------------------------------------- 6.0/124.0 MB 9.3 MB/s eta 0:00:13
   -- ------------------------------------- 8.4/124.0 MB 9.7 MB/s eta 0:00:12
   --- ------------------------------------ 10.5/124.0 MB 9.7 MB/s eta 0:00:12
   --- ------------------------------------ 12.3/124.0 MB 9.7 MB/s eta 0:00:12
   ---- ----------------------------------- 14.4/124.0 MB 9.7 MB/s eta 0:00:12
   ----- ---------------------------------- 16.5/124.0 MB 9.7 MB/s eta 0:00:12
   ----- ---------------------------------- 18.4/124.0 MB 9.7 MB/s eta 0:00:11
   ------ --------------------------------- 20.4/124.0 MB 9.7 MB/s eta 0:00:11
   ------- ----


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import pandas as pd
import torch

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

CLEANED_PATH = r"C:\glassIDS\data\processed\cicids2017_cleaned.csv"

df = pd.read_csv(CLEANED_PATH)

feature_cols = []
for c in df.columns:
    if c not in ("Label", "is_attack", "source_file", "Destination Port"):
        feature_cols.append(c)

X = df[feature_cols]
y = df["Label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

print(X_train.shape, X_test.shape)
print(list(le.classes_))

(2016638, 77) (504160, 77)
['BENIGN', 'Bot', 'DDoS', 'DoS GoldenEye', 'DoS Hulk', 'DoS Slowhttptest', 'DoS slowloris', 'FTP-Patator', 'Heartbleed', 'Infiltration', 'PortScan', 'SSH-Patator', 'Web Attack - Brute Force', 'Web Attack - Sql Injection', 'Web Attack - XSS']


In [7]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(X_train_scaled.mean(axis=0)[:5])
print(X_train_scaled.std(axis=0)[:5])

[-5.41194623e-18  5.63744399e-20  1.69123320e-19  7.58236217e-18
 -3.38246639e-19]
[1. 1. 1. 1. 1.]


In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [10]:
from torch.utils.data import Dataset, DataLoader

class FlowDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = FlowDataset(X_train_scaled, y_train_enc)
test_dataset = FlowDataset(X_test_scaled, y_test_enc)

BATCH_SIZE = 512

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

X_batch, y_batch = next(iter(train_loader))
print(X_batch.shape, X_batch.dtype)
print(y_batch.shape, y_batch.dtype)

torch.Size([512, 77]) torch.float32
torch.Size([512]) torch.int64
